# آیا تخفیف واقعاً باعث فروش تعداد بیشتری از کالا می‌شود؟
### آزمون آماری فرضیه‌ی رایج «فروشگاه‌ها با تخفیف مشتری را فریب می‌دهند» — دیتاست Superstore

**زمینه‌ی موضوع**

باوری رایج در میان مردم وجود دارد که فروشگاه‌ها با اعمال تخفیف، مشتریان را ترغیب می‌کنند
تا حجم بیشتری خرید کنند و در نهایت سود بیشتری کسب می‌کنند. در این نوت‌بوک، بخش اول این
ادعا — یعنی تأثیر تخفیف بر **تعداد کالای فروخته‌شده** — با روش‌های آماری معتبر بررسی می‌شود.

**سؤال پژوهش**

> آیا اعمال تخفیف روی یک قلم کالا، به‌صورت معنادار (هم از نظر آماری و هم از نظر عملی)
> باعث تغییر در تعداد واحدهای فروخته‌شده می‌شود؟

**نقشه‌ی راه تحلیل**

۱. بارگذاری و بررسی اولیه‌ی داده‌ها  
۲. تفکیک داده‌ها به دو گروه: **تخفیف‌دار** (`Discount > 0`) و **بدون تخفیف** (`Discount == 0`)  
۳. بررسی و رسم توزیع متغیر `Quantity` در هر گروه  
۴. بررسی پیش‌فرض‌های آماری لازم (نرمال بودن، همگنی واریانس) پیش از انتخاب آزمون مناسب  
۵. اجرای آزمون(های) فرضیه و محاسبه‌ی اندازه‌ی اثر (نه فقط مقدار p)  
۶. تفسیر نتیجه به زبان کسب‌وکار، همراه با یک بررسی تکمیلی برای درآمد و سود

> **نکته درباره‌ی منبع داده:** این نوت‌بوک انتظار دارد فایل خام سطح سفارش دیتاست
> Superstore (همان فایلی که در فاز آماده‌سازی داده و انبار داده استفاده شده) به‌صورت CSV
> در دسترس باشد. مسیر آن را در متغیر `DATA_PATH` در بخش «بارگذاری داده‌ها» تنظیم کنید.


## ۱. آماده‌سازی و وارد کردن کتابخانه‌ها

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42
ALPHA = 0.05


## ۲. بارگذاری داده‌ها

دیتاست Superstore معمولاً به‌صورت یک فایل CSV با ستون‌های زیر (در میان سایر ستون‌ها) ارائه می‌شود:

| ستون | معنی |
|---|---|
| `Discount` | نرخ تخفیف اعمال‌شده روی هر ردیف سفارش (۰ یعنی بدون تخفیف) |
| `Quantity` | تعداد واحدهای فروخته‌شده در آن ردیف سفارش |
| `Sales` | درآمد حاصل از آن ردیف سفارش |
| `Profit` | سود حاصل از آن ردیف سفارش |
| `Category` / `Sub-Category` | دسته‌بندی محصول |

مسیر `DATA_PATH` را در صورت نیاز با مسیر فایل محلی خود جایگزین کنید.


In [ ]:
DATA_PATH = "data/Sample-Superstore.csv"

try:
    df_raw = pd.read_csv(DATA_PATH, encoding="utf-8")
except UnicodeDecodeError:
    df_raw = pd.read_csv(DATA_PATH, encoding="latin1")

print(f"Rows: {df_raw.shape[0]:,} | Columns: {df_raw.shape[1]}")
df_raw.head()


In [ ]:
df_raw.columns = [c.strip() for c in df_raw.columns]
df_raw.info()


## ۳. اعتبارسنجی و پاک‌سازی داده‌ها

پیش از تفکیک داده‌ها به دو گروه، اطمینان حاصل می‌کنیم که ستون‌های موردنیاز
(`Discount` و `Quantity`) وجود دارند، عددی هستند و مقدار گمشده یا نامعتبر ندارند.


In [ ]:
required_cols = ["Discount", "Quantity"]
missing_cols = [c for c in required_cols if c not in df_raw.columns]
assert not missing_cols, f"Missing required column(s): {missing_cols}"

df = df_raw.copy()

df["Discount"] = pd.to_numeric(df["Discount"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")

n_before = len(df)
df = df.dropna(subset=["Discount", "Quantity"])
df = df[df["Quantity"] >= 0]
n_after = len(df)

print(f"Dropped {n_before - n_after:,} invalid/missing rows out of {n_before:,} "
      f"({(n_before - n_after) / n_before:.2%}).")
print(f"Remaining rows for analysis: {n_after:,}")


## ۴. تفکیک داده‌ها: تخفیف‌دار در برابر بدون تخفیف

هر ردیف سفارش با یکی از دو برچسب زیر مشخص می‌شود:

- **Discounted (تخفیف‌دار):** `Discount > 0`  
- **Full Price (بدون تخفیف):** `Discount == 0`


In [ ]:
df["Discount_Group"] = np.where(df["Discount"] > 0, "Discounted", "Full Price")

group_counts = df["Discount_Group"].value_counts()
group_share = df["Discount_Group"].value_counts(normalize=True)

summary_split = pd.DataFrame({
    "Count": group_counts,
    "Share (%)": (group_share * 100).round(2)
})
summary_split


In [ ]:
discounted = df.loc[df["Discount_Group"] == "Discounted", "Quantity"]
full_price = df.loc[df["Discount_Group"] == "Full Price", "Quantity"]

print(f"Discounted group : n = {len(discounted):,}")
print(f"Full price group : n = {len(full_price):,}")


## ۵. آمار توصیفی متغیر `Quantity` به تفکیک گروه

پیش از اجرای هر آزمون رسمی، شاخص‌های مرکزی، پراکندگی و شکل توزیع را در دو گروه مقایسه می‌کنیم.


In [ ]:
def describe_group(series: pd.Series) -> pd.Series:
    return pd.Series({
        "count": series.count(),
        "mean": series.mean(),
        "median": series.median(),
        "std": series.std(),
        "min": series.min(),
        "25%": series.quantile(0.25),
        "75%": series.quantile(0.75),
        "max": series.max(),
        "skewness": series.skew(),
        "kurtosis": series.kurt(),
    })

desc_table = pd.DataFrame({
    "Discounted": describe_group(discounted),
    "Full Price": describe_group(full_price),
})
desc_table


## ۶. نمایش بصری توزیع‌ها

هیستوگرام/KDE و باکس‌پلات برای بررسی شکل، پراکندگی و مقادیر پرت در کنار هم رسم می‌شوند.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(
    data=df, x="Quantity", hue="Discount_Group", stat="density",
    common_norm=False, kde=True, element="step", alpha=0.35, ax=axes[0]
)
axes[0].set_title("Distribution of Quantity Sold by Discount Group")
axes[0].set_xlabel("Quantity per Order Line")

sns.boxplot(data=df, x="Discount_Group", y="Quantity", ax=axes[1], showmeans=True)
axes[1].set_title("Quantity Sold by Discount Group (Boxplot)")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
sns.violinplot(data=df, x="Discount_Group", y="Quantity", inner="quartile")
plt.title("Quantity Distribution Shape by Discount Group")
plt.xlabel("")
plt.tight_layout()
plt.show()


## ۷. بررسی پیش‌فرض‌های آزمون آماری

متغیر `Quantity` یک متغیر شمارشی، گسسته و با چولگی به سمت راست است؛ بنابراین نباید
پیش‌فرض نرمال بودن آن را بدون بررسی پذیرفت. پیش از انتخاب بین آزمون پارامتری
(t-test) و معادل ناپارامتری آن (Mann-Whitney U)، این پیش‌فرض‌ها را به‌صورت رسمی بررسی می‌کنیم:

- **نرمال بودن:** نمودار Q-Q (بصری) + آزمون D'Agostino-Pearson (عددی؛ برای نمونه‌های بزرگ
  قابل‌اعتمادتر از Shapiro-Wilk است).
- **همگنی واریانس:** آزمون Levene (نسبت به غیرنرمال بودن داده مقاوم است).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
stats.probplot(discounted, dist="norm", plot=axes[0])
axes[0].set_title("Q-Q Plot — Discounted Group")

stats.probplot(full_price, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot — Full Price Group")

plt.tight_layout()
plt.show()


In [ ]:
normality_results = {}
for name, series in [("Discounted", discounted), ("Full Price", full_price)]:
    stat, p_value = stats.normaltest(series)
    normality_results[name] = {"statistic": stat, "p_value": p_value,
                                "normal_at_5pct": p_value > ALPHA}

normality_df = pd.DataFrame(normality_results).T
normality_df


In [ ]:
levene_stat, levene_p = stats.levene(discounted, full_price)
print(f"Levene's test statistic = {levene_stat:.4f}, p-value = {levene_p:.4g}")
print("Equal variances assumption holds at 5%:", levene_p > ALPHA)


**تفسیر بررسی پیش‌فرض‌ها**

از آنجا که `Quantity` یک متغیر گسسته، با دامنه‌ی کوچک و چوله به راست است، انتظار می‌رود
آزمون نرمال بودن و نمودارهای Q-Q در هر دو گروه، فرض نرمال بودن را رد کنند. این وضعیت برای
این نوع داده‌ی شمارشی طبیعی است و دقیقاً به همین دلیل، **آزمون Mann-Whitney U** به‌عنوان
آزمون اصلی فرضیه در ادامه استفاده می‌شود و **آزمون t مستقل** تنها به‌عنوان یک بررسی
تکمیلی و اطمینان‌بخش (با استناد به قضیه‌ی حد مرکزی برای نمونه‌های بزرگ) به کار می‌رود.


## ۸. تعریف فرضیه‌ها

- **فرضیه‌ی صفر (H0):** توزیع تعداد `Quantity` فروخته‌شده در دو گروه تخفیف‌دار و
  بدون‌تخفیف یکسان است — تخفیف تأثیری بر تعداد فروش ندارد.
- **فرضیه‌ی مقابل (H1):** توزیع تعداد `Quantity` فروخته‌شده بین دو گروه متفاوت است.

سطح معناداری آزمون: **α = 0.05**


## ۹. آزمون اصلی — Mann-Whitney U (ناپارامتری)

از آنجا که `Quantity` نرمال نیست، آزمون Mann-Whitney U انتخاب مناسبی است: این آزمون
بدون فرض توزیع خاصی، دو گروه را مقایسه می‌کند و بررسی می‌کند که آیا یکی از گروه‌ها به‌طور
سیستماتیک مقادیر بالاتری دارد یا نه.


In [ ]:
u_stat, u_p_value = stats.mannwhitneyu(discounted, full_price, alternative="two-sided")

print(f"Mann-Whitney U statistic = {u_stat:,.1f}")
print(f"p-value = {u_p_value:.4g}")

if u_p_value < ALPHA:
    print(f"Result: Reject H0 at alpha = {ALPHA} — the two distributions differ significantly.")
else:
    print(f"Result: Fail to reject H0 at alpha = {ALPHA} — no significant difference detected.")


In [ ]:
n1, n2 = len(discounted), len(full_price)
rank_biserial = 1 - (2 * u_stat) / (n1 * n2)

print(f"Rank-biserial correlation (effect size) = {rank_biserial:.4f}")
print("Interpretation guide: |r| < 0.1 negligible | 0.1-0.3 small | 0.3-0.5 medium | >= 0.5 large")


## ۱۰. بررسی تکمیلی — آزمون t ولش (Welch)

به‌عنوان یک بررسی مکمل (نه نتیجه‌ی اصلی)، آزمون t ولش را نیز اجرا می‌کنیم که فرض برابری
واریانس‌ها را ندارد. با توجه به حجم بالای نمونه، توزیع نمونه‌ای میانگین‌های دو گروه طبق
قضیه‌ی حد مرکزی تقریباً نرمال خواهد بود؛ بنابراین آزمون t نیز می‌تواند به‌عنوان یک بررسی
ثانویه‌ی معقول در کنار آزمون اصلی به کار رود.


In [ ]:
t_stat, t_p_value = stats.ttest_ind(discounted, full_price, equal_var=False)

print(f"Welch's t-statistic = {t_stat:.4f}")
print(f"p-value = {t_p_value:.4g}")

if t_p_value < ALPHA:
    print(f"Result: Reject H0 at alpha = {ALPHA} — mean quantities differ significantly.")
else:
    print(f"Result: Fail to reject H0 at alpha = {ALPHA} — no significant difference in means.")


In [ ]:
mean_diff = discounted.mean() - full_price.mean()
pooled_std = np.sqrt(
    ((n1 - 1) * discounted.std(ddof=1) ** 2 + (n2 - 1) * full_price.std(ddof=1) ** 2)
    / (n1 + n2 - 2)
)
cohens_d = mean_diff / pooled_std

print(f"Mean quantity (Discounted) = {discounted.mean():.4f}")
print(f"Mean quantity (Full Price) = {full_price.mean():.4f}")
print(f"Mean difference            = {mean_diff:.4f}")
print(f"Cohen's d (effect size)    = {cohens_d:.4f}")
print("Interpretation guide: |d| < 0.2 negligible | 0.2-0.5 small | 0.5-0.8 medium | >= 0.8 large")


## ۱۱. جمع‌بندی نتایج آزمون‌ها

In [ ]:
results_summary = pd.DataFrame([
    {
        "Test": "Mann-Whitney U (primary)",
        "Statistic": u_stat,
        "p-value": u_p_value,
        "Significant at 5%": u_p_value < ALPHA,
        "Effect size": rank_biserial,
        "Effect size metric": "Rank-biserial r",
    },
    {
        "Test": "Welch's t-test (robustness check)",
        "Statistic": t_stat,
        "p-value": t_p_value,
        "Significant at 5%": t_p_value < ALPHA,
        "Effect size": cohens_d,
        "Effect size metric": "Cohen's d",
    },
])
results_summary


## ۱۲. بررسی تکمیلی کسب‌وکاری — درآمد و سود

ادعای رایج تنها درباره‌ی تعداد فروش نیست، بلکه می‌گوید فروشگاه‌ها «در نهایت پول بیشتری
درمی‌آورند». به‌عنوان یک بررسی مکمل (و نه بخش اصلی تحلیل)، در صورت وجود ستون‌های
`Sales` و `Profit`، مجموع و میانگین آن‌ها را به تفکیک گروه بررسی می‌کنیم.


In [ ]:
extra_cols = [c for c in ["Sales", "Profit"] if c in df.columns]

if extra_cols:
    business_check = df.groupby("Discount_Group")[extra_cols + ["Quantity"]].agg(
        ["sum", "mean"]
    )
    business_check
else:
    print("Sales/Profit columns not found in this dataset — skipping the business sanity check.")


In [ ]:
if extra_cols:
    fig, axes = plt.subplots(1, len(extra_cols), figsize=(6 * len(extra_cols), 5))
    if len(extra_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, extra_cols):
        sns.barplot(data=df, x="Discount_Group", y=col, estimator=np.mean, ax=ax, errorbar="sd")
        ax.set_title(f"Average {col} by Discount Group")
        ax.set_xlabel("")
    plt.tight_layout()
    plt.show()


## ۱۳. نتیجه‌گیری

پس از اجرای نوت‌بوک روی داده‌ی واقعی، این بخش را با استناد به مقادیر عددی به‌دست‌آمده در
بالا تکمیل کنید. الگویی برای جمع‌بندی در ادامه آمده است.

**نتیجه‌گیری آماری**

- آزمون Mann-Whitney U در سطح α = 0.05 فرضیه‌ی صفر را [رد می‌کند / رد نمی‌کند]
  (p = *مقدار*)، که [با / بدون] تأیید از سوی آزمون t ولش (p = *مقدار*) همراه است.
- اندازه‌ی اثر (rank-biserial r = *مقدار*، Cohen's d = *مقدار*) نشان‌دهنده‌ی یک تفاوت
  عملی [ناچیز / کوچک / متوسط / بزرگ] در تعداد فروش بین دو گروه تخفیف‌دار و بدون‌تخفیف است.

**تفسیر کسب‌وکاری**

- معناداری آماری و معناداری عملی یک چیز نیستند. در دیتاست‌های بزرگ تجارت الکترونیک، حتی
  یک تفاوت بسیار کوچک و بی‌اهمیت از نظر تجاری نیز می‌تواند مقدار p بسیار کوچکی تولید کند.
  اعداد اندازه‌ی اثر که در بالا محاسبه شدند، باید در جمع‌بندی کسب‌وکاری وزن بیشتری نسبت
  به تنهایِ مقدار p داشته باشند.
- اگر اندازه‌ی اثر به‌دست‌آمده کوچک یا ناچیز باشد، این ادعای رایج که «تخفیف‌ها مردم را
  به خرید بسیار بیشتر ترغیب می‌کنند»، **با شواهد این داده تأیید نمی‌شود** — حتی اگر تفاوت
  از نظر آماری معنادار باشد، ممکن است از نظر عملی چندان چشمگیر نباشد.
- بررسی تکمیلی درآمد و سود کمک می‌کند دو ادعای جداگانه از هم تفکیک شوند: (۱) تخفیف باعث
  فروش تعداد بیشتری کالا می‌شود، و (۲) تخفیف باعث درآمد بیشتر می‌شود. این دو می‌توانند در
  جهت‌های مخالف حرکت کنند (برای مثال، فروش تعداد بیشتر با حاشیه‌ی سود پایین‌تر می‌تواند
  در نهایت سود کل را کاهش دهد).

> جای‌گزین‌های داخل کروشه را پس از اجرای واقعی این نوت‌بوک روی داده‌ی اصلی، با مقادیر
> واقعی جایگزین کنید.
